# MLflow: Experiment Tracking and Model Management

## The Core Question

You run 20 hyperparameter combinations for your turbulence model. Three weeks later, a colleague asks: "What was the best model, what data did it use, and what were the parameters?" You have 20 Jupyter notebooks with `print` statements. Can you answer in 5 minutes?

**MLflow** is a platform for managing the ML lifecycle: **experiment tracking** (parameters, metrics, artifacts), **model registry** (version, stage, share), and **deployment** (skip here). The core value for an engineer: reproducible experiments with a queryable history.

## Why MLflow?

| Problem | Without MLflow | With MLflow |
|---------|---------------|-------------|
| "Which run was best?" | Dig through notebooks | `mlflow ui` -> sort by metric |
| Reproduce a result | Hope you remember | Click run -> see exact params, env, git commit |
| Share a model | "Here's the .pkl" | Registry: named, versioned, staged |
| Compare runs | Side-by-side notebooks | Built-in comparison view |

## The Workflow

```
Code (train.py)
    |
    v
mlflow.start_run()
    |
    +-- mlflow.log_param('lr', 0.01)
    +-- mlflow.log_metric('rmse', 0.042)
    +-- mlflow.log_artifact('model.pkl')
    |
    v
mlflow.end_run()
    |
    v
Tracking Server (local file or MLflow server)
    |
    v
mlflow ui  ->  browse, compare, load models


In [ ]:
# Install and set up MLflow with local tracking
!pip install mlflow -q 2>/dev/null

import mlflow
import mlflow.sklearn
import os

# Use local directory as tracking store (no server needed)
os.environ['MLFLOW_TRACKING_URI'] = 'file:./mlruns'
mlflow.set_tracking_uri('file:./mlruns')
mlflow.set_experiment('turbulence-regression')

print(f'Tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment: {mlflow.active_experiment().name}')
print('All logged data goes to ./mlruns/')


## 1. Manual Experiment Logging

**Concept**: The basic pattern: start a run, log params/metrics/artifacts, end the run. Each run is one training attempt.


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Synthetic CFD-inspired data: predict pressure from flow parameters
np.random.seed(42)
n = 2000
Re = np.random.uniform(1e5, 1e7, n)          # Reynolds number
alpha = np.random.uniform(5, 30, n)           # Angle of attack
Mach = np.random.uniform(0.3, 1.5, n)         # Mach number
Cd = 0.01 + 0.005 * np.sin(alpha) + 0.001 * (Mach - 0.7)**2 + 0.0001 * (Re - 5e5)/1e5 + np.random.randn(n) * 0.002

X = np.column_stack([np.log10(Re), alpha, Mach])
y = Cd
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Log a single run manually ---
with mlflow.start_run(run_name='baseline-rf'):
    params = {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 5}
    
    for k, v in params.items():
        mlflow.log_param(k, v)
    
    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = model.score(X_test, y_test)
    
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('r2', r2)
    
    # Log the model as an artifact
    import joblib
    with mlflow.start_run():
        pass  # (we're already in a run, this is just showing the pattern)
    
    print(f'RMSE: {rmse:.5f}  R2: {r2:.4f}')


## 2. Hyperparameter Sweep (Manual Loop)

**Concept**: Run multiple configurations, each as a separate MLflow run. Compare them later in the UI. This is the most common engineering use case.


In [ ]:
import itertools

n_est_range = [50, 100, 200, 500]
depth_range = [5, 10, 20, None]
min_split_range = [2, 5, 10]

# Full grid = 4*4*3 = 48 runs (fast for RF on small data)
results = []

for n_est, depth, min_split in itertools.product(n_est_range, depth_range, min_split_range):
    with mlflow.start_run(run_name=f'n={n_est}_d={depth}_m={min_split}'):
        params = {'n_estimators': n_est, 'max_depth': depth, 'min_samples_split': min_split}
        
        for k, v in params.items():
            mlflow.log_param(k, v)
        
        model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = model.score(X_test, y_test)
        
        mlflow.log_metric('rmse', rmse)
        mlflow.log_metric('r2', r2)
        
        results.append({'n_est': n_est, 'depth': depth, 'min_split': min_split, 'rmse': rmse})

print(f'Completed {len(results)} runs')
print(f'Best RMSE: {min(r["rmse"] for r in results):.5f}')


## 3. Auto-Logging with MLflow + scikit-learn

**Concept**: MLflow can auto-log everything (params, metrics, model) with one line. No manual `log_param` calls. Works with sklearn, XGBoost, PyTorch, and more.


In [ ]:
from xgboost import XGBRegressor

# Enable auto-logging
mlflow.sklearn.autolog()

with mlflow.start_run(run_name='xgboost-autolog'):
    xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                        random_state=42, n_jobs=-1)
    xgb.fit(X_train, y_train)
    # That's it. MLflow logged params, metrics, and the model automatically.

print('XGBoost run logged automatically (params + metrics + model).')
print('Check the MLflow UI to see the logged parameters and metrics.')


## 4. Viewing Results in the MLflow UI

**Concept**: Launch the local MLflow server to browse all runs, compare metrics, and load models.


In [ ]:
# Launch MLflow UI (runs in background)
import subprocess
import time

proc = subprocess.Popen(
    ['mlflow', 'ui', '--port', '5000', '--backend-store-uri', 'file:./mlruns'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

print('MLflow UI launched at: http://localhost:5000')
print('\nIn the UI you can:')
print('  - View all 48+ runs in the Experiment view')
print('  - Sort by RMSE to find the best configuration')
print('  - Compare any two runs side-by-side')
print('  - Download logged models as artifacts')
print('  - See exact parameters for each run')
print('\nNOTE: This is a background process. It will keep running.')
print('To stop it:  proc.terminate()')


## 5. Querying Runs Programmatically

**Concept**: You don't need the UI. Query runs via Python API to find the best configuration, load the model, or export results.


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri='file:./mlruns')

# Get the experiment
exp = client.get_experiment_by_name('turbulence-regression')
print(f'Experiment: {exp.name} (id={exp.experiment_id})')

# Get all runs, sorted by RMSE (ascending = best first)
runs = client.search_runs([exp.experiment_id], order_by=["metrics.rmse ASC"])

print(f'\nTotal runs: {len(runs)}')
print('\nTop 5 by RMSE:')
print(f'{"Run Name":<35s} {"RMSE":>10s} {"n_est":>6s} {"depth":>6s}')
print('-' * 60)

for run in runs[:5]:
    rmse = run.metrics.get('rmse', float('nan'))
    n_est = run.params.get('n_estimators', '?')
    depth = run.params.get('max_depth', '?')
    print(f'{run.info.run_name:<35s} {rmse:>10.5f} {str(n_est):>6s} {str(depth):>6s}')

# Load the best model from the artifact store
best_run = runs[0]
print(f'\nBest run: {best_run.info.run_name}')
print(f'Artifacts: {best_run.data_tags.keys() if best_run.data_tags else "(none)"}')


## 6. Model Registry (Versioning & Staging)

**Concept**: The registry lets you name, version, and stage models (`Staging` -> `Production`). For a single engineer, the experiment tracking (sections 1-5) is the main value. The registry matters when you have multiple people deploying models.


In [ ]:
# Register the best model in the registry
# (Requires a registry-compatible store; file-based works for demo)

with mlflow.start_run(run_name='register-best'):
    # Re-train the best config
    best_params = runs[0].params
    best_model = RandomForestRegressor(
        n_estimators=int(best_params.get('n_estimators', 100)),
        max_depth=best_params.get('max_depth', 'None') if best_params.get('max_depth', 'None') != 'None' else None,
        min_samples_split=int(best_params.get('min_samples_split', 5)),
        random_state=42, n_jobs=-1
    )
    best_model.fit(X_train, y_train)
    
    mlflow.sklearn.log_model(best_model, artifact_path='model')
    model_id = mlflow.sklearn.log_model(best_model, artifact_path='model').model_uuid

    # Create a registered model and version
    try:
        client.create_registered_model('turbulence-cd-predictor')
    except Exception:
        pass  # already exists
    
    mv = client.create_model_version(
        name='turbulence-cd-predictor',
        source=best_model,  # This will fail for file-based; use run info instead
        run_id=best_run.info.run_id
    )
    print(f'Registered model: {mv.name} v{mv.version}')

print('\nNOTE: Full registry functionality requires a database-backed store (Postgres, etc.).')
print('For personal use, the experiment tracking (sections 1-5) is the main value.')


## Practical Tips

| Situation | Recommendation |
|-----------|---------------|
| Personal experimentation | `file:./mlruns` (no server, no setup) |
| Team sharing | `mlflow server` on a shared machine + Postgres |
| Cloud | Databricks/MLflow on AWS/GCP |
| CI/CD | Log from `pytest` or `make train`; query in CI to gate on metrics |
| HPC jobs | Set `MLFLOW_TRACKING_URI` env var; log from job scripts |
| Reproducibility | Log git commit hash: `mlflow.log_param('git_commit', subprocess.check_output(['git','rev-parse','HEAD']).decode().strip())` |

**Rule of thumb**: If you run more than 3 experiments with different parameters, use MLflow. The tracking overhead is one `with mlflow.start_run()` block.


## Exercises

1. **Extend the sweep**: Add `max_features` as a 4th parameter. Run 100+ configurations. Sort by RMSE in the UI. What fraction of the grid is within 5% of the best?

2. **Log artifacts**: Save a plot of predicted vs. actual Cd for the best run. `mlflow.log_figure(fig, 'cd_scatter.png')`. View it in the UI.

3. **Compare frameworks**: Train the same data with sklearn RF, XGBoost, and LightGBM. Log all three. Which wins? Is the difference statistically significant (run 5 seeds each)?

4. **Git integration**: Log `git rev-parse HEAD` as a parameter in each run. After a code change, run 2 more experiments. Can you trace which code version produced which result?

5. **Integrate with your CFD workflow**: Wrap your existing parameter study (PyDOE or SALib) in an MLflow run. Log the DOE design matrix as an artifact. Log the best parameter set. Can you reproduce the best result in one click?
